# Lab 7B: Build Your Own MCP Server

Deploy an MCP server using Azure Functions that calculates ISS position.

This uses the `mcpToolTrigger` binding - no HTTP-level MCP implementation needed!

In [7]:
# Enable automatic masking of Azure resource names in print output
import sys
sys.path.insert(0, "../../")
from secure_print import install
install()

secure_print: Azure resource masking enabled


## Part 1: Configuration

In [ ]:
import os, json, subprocess, math, hashlib, requests
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv("../../.env")

RESOURCE_GROUP = "lab7b-byo-mcp"
LOCATION = "eastus2"

# Use deterministic hash (subscription + RG) for cross-user uniqueness
def get_unique_suffix():
    result = subprocess.run("az account show --query id -o tsv", capture_output=True, text=True, shell=True)
    sub_id = result.stdout.strip() if result.returncode == 0 else "default"
    return hashlib.md5(f"{sub_id}-{RESOURCE_GROUP}".encode()).hexdigest()[:6]

UNIQUE_SUFFIX = os.environ.get("UNIQUE_SUFFIX", get_unique_suffix())
FUNC_APP_NAME = f"iss-mcp-{UNIQUE_SUFFIX}"
STORAGE_NAME = f"issmcp{UNIQUE_SUFFIX}".replace("-", "")[:24]

SPOKE_ENDPOINT = os.environ.get("SPOKE_ENDPOINT", "")
SPOKE_PROJECT = os.environ.get("SPOKE_PROJECT", "")
APIM_CONNECTION = os.environ.get("APIM_CONNECTION", "")
MODEL_NAME = os.environ.get("MODEL_NAME", "gpt-4o")

if SPOKE_ENDPOINT:
    account_host = SPOKE_ENDPOINT.replace("https://", "").replace(".cognitiveservices.azure.com/", "")
    PROJECT_ENDPOINT = f"https://{account_host}.services.ai.azure.com/api/projects/{SPOKE_PROJECT}"
    GATEWAY_MODEL = f"{APIM_CONNECTION}/{MODEL_NAME}"
else:
    PROJECT_ENDPOINT = ""
    GATEWAY_MODEL = ""

result = subprocess.run("az account show -o json", capture_output=True, text=True, shell=True)
SUBSCRIPTION_ID = json.loads(result.stdout)["id"] if result.returncode == 0 and result.stdout.strip().startswith("{") else None

print(f"Function App: {FUNC_APP_NAME}")
print(f"Storage:      {STORAGE_NAME}")


## Part 2: Create MCP Server with Built-in Triggers

Using `mcpToolTrigger` - the tools are defined via decorators, not HTTP handlers!

In [9]:
import os
os.makedirs("iss-mcp", exist_ok=True)
print("Created: iss-mcp/")

Created: iss-mcp/


In [10]:
%%writefile iss-mcp/requirements.txt
azure-functions

Overwriting iss-mcp/requirements.txt


In [11]:
%%writefile iss-mcp/host.json
{
  "version": "2.0",
  "extensionBundle": {
    "id": "Microsoft.Azure.Functions.ExtensionBundle.Experimental",
    "version": "[4.*, 5.0.0)"
  }
}

Overwriting iss-mcp/host.json


In [12]:
%%writefile iss-mcp/function_app.py
import json
import math
from datetime import datetime, timezone
import azure.functions as func

app = func.FunctionApp(http_auth_level=func.AuthLevel.FUNCTION)

# ISS Orbital Constants
EARTH_RADIUS_KM = 6371.0
ISS_ALTITUDE_KM = 408.0
ISS_ORBITAL_PERIOD_MIN = 92.65
ISS_INCLINATION_DEG = 51.6
ISS_VELOCITY_KMS = 7.66
REFERENCE_EPOCH = datetime(2026, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
REFERENCE_LONGITUDE = -80.0

def get_cardinal(h):
    return ["N","NE","E","SE","S","SW","W","NW"][round(h/45)%8]

def calc_position():
    now = datetime.now(timezone.utc)
    mins = (now - REFERENCE_EPOCH).total_seconds() / 60.0
    phase = (mins / ISS_ORBITAL_PERIOD_MIN % 1.0) * 2 * math.pi
    inc = math.radians(ISS_INCLINATION_DEG)
    lat = math.degrees(math.asin(math.sin(inc) * math.sin(phase)))
    drift = (360/ISS_ORBITAL_PERIOD_MIN - 360/1436) * mins
    lon_off = math.degrees(math.atan2(math.cos(inc)*math.sin(phase), math.cos(phase)))
    lon = (REFERENCE_LONGITUDE - drift + lon_off) % 360
    if lon > 180: lon -= 360
    return lat, lon, now, phase

# Tool 1: Get ISS Position
@app.generic_trigger(
    arg_name="context",
    type="mcpToolTrigger",
    toolName="get_iss_position",
    description="Get the current position of the International Space Station",
    toolProperties="[]"
)
def get_iss_position(context) -> str:
    lat, lon, now, _ = calc_position()
    return json.dumps({
        "latitude": round(lat, 4),
        "longitude": round(lon, 4),
        "altitude_km": ISS_ALTITUDE_KM,
        "timestamp": now.isoformat()
    })

# Tool 2: Get ISS Velocity
@app.generic_trigger(
    arg_name="context",
    type="mcpToolTrigger",
    toolName="get_iss_velocity",
    description="Get the current velocity and heading of the ISS",
    toolProperties="[]"
)
def get_iss_velocity(context) -> str:
    lat, _, _, phase = calc_position()
    lat_r, inc_r = math.radians(lat), math.radians(ISS_INCLINATION_DEG)
    if abs(lat) < ISS_INCLINATION_DEG:
        heading = math.degrees(math.acos(max(-1, min(1, math.cos(lat_r)/math.cos(inc_r)))))
        if math.pi/2 < phase < 3*math.pi/2: heading = 180 - heading
    else:
        heading = 90 if lat > 0 else 270
    return json.dumps({
        "velocity_kms": ISS_VELOCITY_KMS,
        "velocity_mph": round(ISS_VELOCITY_KMS * 2236.94),
        "heading_degrees": round(heading, 1),
        "direction": get_cardinal(heading)
    })

# Tool 3: Get Orbital Info
@app.generic_trigger(
    arg_name="context",
    type="mcpToolTrigger",
    toolName="get_orbital_info",
    description="Get ISS orbital parameters",
    toolProperties="[]"
)
def get_orbital_info(context) -> str:
    return json.dumps({
        "altitude_km": ISS_ALTITUDE_KM,
        "orbital_period_minutes": ISS_ORBITAL_PERIOD_MIN,
        "inclination_degrees": ISS_INCLINATION_DEG,
        "velocity_kms": ISS_VELOCITY_KMS,
        "orbits_per_day": round(1440 / ISS_ORBITAL_PERIOD_MIN, 2)
    })

# Tool 4: Check Visibility
visibility_props = json.dumps([
    {"propertyName": "latitude", "propertyType": "number", "description": "Observer latitude"},
    {"propertyName": "longitude", "propertyType": "number", "description": "Observer longitude"}
])

@app.generic_trigger(
    arg_name="context",
    type="mcpToolTrigger",
    toolName="get_iss_visibility",
    description="Check if the ISS is visible from a given location",
    toolProperties=visibility_props
)
def get_iss_visibility(context) -> str:
    args = json.loads(context).get("arguments", {})
    obs_lat = args.get("latitude", 0)
    obs_lon = args.get("longitude", 0)
    
    iss_lat, iss_lon, _, _ = calc_position()
    lat1, lat2 = math.radians(obs_lat), math.radians(iss_lat)
    dlon = math.radians(iss_lon - obs_lon)
    cos_d = max(-1, min(1, math.sin(lat1)*math.sin(lat2) + math.cos(lat1)*math.cos(lat2)*math.cos(dlon)))
    dist = math.acos(cos_d) * EARTH_RADIUS_KM
    bearing = (math.degrees(math.atan2(
        math.sin(dlon)*math.cos(lat2),
        math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    )) + 360) % 360
    
    return json.dumps({
        "is_visible": dist < 2500,
        "distance_km": round(dist, 1),
        "direction": get_cardinal(bearing),
        "iss_latitude": round(iss_lat, 4),
        "iss_longitude": round(iss_lon, 4)
    })

Overwriting iss-mcp/function_app.py


## Part 3: Deploy to Azure

### deploy resource group

In [13]:
# Create resource group (Windows-compatible)
result = subprocess.run(
    f'az group create --name {RESOURCE_GROUP} --location {LOCATION} -o none',
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    print(f"✅ Resource group ready: {RESOURCE_GROUP}")
else:
    print(f"❌ Failed to create resource group: {result.stderr}")

✅ Resource group ready: lab7b-byo-mcp


### deploy storage account (Entra Auth only)

In [14]:
# Create storage account (Entra-based authentication, no shared keys)
result = subprocess.run(
    f'az storage account show -g "{RESOURCE_GROUP}" -n "{STORAGE_NAME}" -o none',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(f"🔧 Creating storage account: {STORAGE_NAME}...")
    subprocess.run(
        f'az storage account create -g "{RESOURCE_GROUP}" -n "{STORAGE_NAME}" -l "{LOCATION}" '
        f'--sku Standard_LRS --allow-shared-key-access false -o none',
        shell=True, capture_output=True, text=True
    )
    print(f"✅ Storage account created (Entra auth only)")
else:
    print(f"✅ Storage account exists: {STORAGE_NAME}")

🔧 Creating storage account: issmcp31d66e...
✅ Storage account created (Entra auth only)


### deploy Azure Function App

In [ ]:
# Create Function App with Entra-based storage authentication
import time

result = subprocess.run(
    f'az functionapp show -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" -o none',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(f"🔧 Creating Function App: {FUNC_APP_NAME}...")
    
    # First, assign current user as Storage Blob Data Contributor to create containers
    result = subprocess.run('az ad signed-in-user show --query id -o tsv', shell=True, capture_output=True, text=True)
    current_user_id = result.stdout.strip()
    storage_scope = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.Storage/storageAccounts/{STORAGE_NAME}"
    
    if current_user_id:
        print("   Assigning storage roles to current user...")
        subprocess.run(f'az role assignment create --assignee "{current_user_id}" --role "Storage Blob Data Contributor" --scope "{storage_scope}" -o none',
                       shell=True, capture_output=True, text=True)
        time.sleep(10)  # Wait for role propagation
    
    # Create deployment container using Entra auth
    print("   Creating deployment container...")
    subprocess.run(
        f'az storage container create --account-name "{STORAGE_NAME}" --name deployments --auth-mode login -o none',
        shell=True, capture_output=True, text=True
    )
    
    # Create Flex Consumption Function App with managed identity storage
    print("   Creating Flex Consumption Function App...")
    subprocess.run(
        f'az functionapp create -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" '
        f'--storage-account "{STORAGE_NAME}" --runtime python --runtime-version 3.11 '
        f'--flexconsumption-location "{LOCATION}" --deployment-storage-container-name deployments '
        f'--deployment-storage-auth-type SystemAssignedIdentity -o none',
        shell=True, capture_output=True, text=True
    )
    print("✅ Function App created")
    
    # Get managed identity principal ID
    result = subprocess.run(
        f'az functionapp identity show -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" --query principalId -o tsv',
        shell=True, capture_output=True, text=True
    )
    principal_id = result.stdout.strip()
    
    if principal_id:
        # Assign all required storage roles to Function App managed identity
        print("   Assigning storage roles to Function App identity...")
        roles = [
            "Storage Blob Data Owner",        # Full blob access for deployments
            "Storage Queue Data Contributor", # Queue access for triggers
            "Storage Table Data Contributor"  # Table access for leases
        ]
        for role in roles:
            # Use --assignee-object-id with --assignee-principal-type for managed identities
            subprocess.run(f'az role assignment create --assignee-object-id "{principal_id}" --assignee-principal-type ServicePrincipal --role "{role}" --scope "{storage_scope}" -o none',
                           shell=True, capture_output=True, text=True)
        print("✅ Storage roles assigned to managed identity")
    
    # Remove default connection string (uses account key) and configure managed identity
    print("   Configuring managed identity storage settings...")
    # First, delete the auto-generated AzureWebJobsStorage connection string (key-based)
    subprocess.run(
        f'az functionapp config appsettings delete -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" --setting-names AzureWebJobsStorage -o none',
        shell=True, capture_output=True, text=True
    )
    # Then set managed identity settings
    settings = [
        f'AzureWebJobsStorage__accountName={STORAGE_NAME}',
        f'AzureWebJobsStorage__credential=managedidentity',
    ]
    subprocess.run(
        f'az functionapp config appsettings set -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" --settings {" ".join(settings)} -o none',
        shell=True, capture_output=True, text=True
    )
    print("✅ Managed identity storage configured")
    
    print("⏳ Waiting 30s for role propagation and initialization...")
    time.sleep(30)
    print(f"✅ Function App ready: {FUNC_APP_NAME}")
else:
    print(f"✅ Function App exists: {FUNC_APP_NAME}")

### deploy the function code

In [16]:
# Deploy Function App code (Windows-compatible)
import zipfile
import time

# Create zip file using Python (cross-platform)
zip_path = "iss-mcp.zip"
source_dir = "iss-mcp"

print(f"📦 Creating deployment package...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, source_dir)
            zipf.write(file_path, arcname)
            print(f"   Added: {arcname}")

print(f"✅ Package created: {zip_path}")

# Deploy to Azure (ignore health check errors - they're expected with managed identity)
print(f"🚀 Deploying to {FUNC_APP_NAME}...")
result = subprocess.run(
    f'az functionapp deployment source config-zip -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" --src "{zip_path}" 2>&1',
    shell=True, capture_output=True, text=True
)

# Check for actual deployment success (status 202 is success)
output = result.stdout + result.stderr
if "status code 202" in output or result.returncode == 0:
    print("✅ Deployment submitted successfully!")
    print("⏳ Waiting 60s for Function App to initialize...")
    time.sleep(60)
    print("✅ Deployment complete!")
else:
    print(f"❌ Deployment failed: {output}")

📦 Creating deployment package...
   Added: host.json
   Added: requirements.txt
   Added: function_app.py
✅ Package created: iss-mcp.zip
🚀 Deploying to iss-mcp-31d66e...
✅ Deployment submitted successfully!
⏳ Waiting 60s for Function App to initialize...
✅ Deployment complete!


In [ ]:
FUNC_HOST = f"{FUNC_APP_NAME}.azurewebsites.net"
result = subprocess.run(
    f'az functionapp keys list -g "{RESOURCE_GROUP}" -n "{FUNC_APP_NAME}" --query masterKey -o tsv',
    shell=True, capture_output=True, text=True
)
FUNC_KEY = result.stdout.strip()

# Built-in MCP endpoint path
MCP_URL = f"https://{FUNC_HOST}/runtime/webhooks/mcp"

print(f"MCP URL: {MCP_URL}")
print(f"Key: {FUNC_KEY[:20]}..." if FUNC_KEY else "Key not available yet")

## Part 4: Test the MCP Server

The built-in MCP endpoint is at `/runtime/webhooks/mcp/sse` for SSE connections.

In [18]:
# Test the SSE endpoint info
sse_url = f"{MCP_URL}/sse?code={FUNC_KEY}"
print(f"SSE Endpoint: {sse_url[:80]}...")
print(f"\nUse this URL with MCP Inspector or Foundry agents")

SSE Endpoint: https://iss***.azurewebsites.net/runtime/webhooks/mcp/sse?code=SEYXmGhbx...

Use this URL with MCP Inspector or Foundry agents


## Part 5: Connect to Foundry Agent

In [19]:
!pip install azure-ai-projects azure-ai-agents azure-identity -q

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.identity import DefaultAzureCredential
from openai.types.responses.response_input_param import McpApprovalResponse

MCP_SSE_URL = f"{MCP_URL}/sse?code={FUNC_KEY}"
print(f"MCP SSE: {MCP_SSE_URL[:60]}...")


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
MCP SSE: https://iss***.azurewebsites.net/runtime/webhooks/mc...


In [20]:
project_client = AIProjectClient(credential=DefaultAzureCredential(), endpoint=PROJECT_ENDPOINT)
openai_client = project_client.get_openai_client()

agent = project_client.agents.create_version(
    agent_name="iss-tracker",
    definition=PromptAgentDefinition(
        model=GATEWAY_MODEL,
        instructions="You track the ISS. Use the MCP tools to answer questions about ISS position, velocity, visibility, and orbital parameters.",
        tools=[MCPTool(server_label="iss_mcp", server_url=MCP_SSE_URL, require_approval="always")],
    )
)
print(f"Agent: {agent.name}")

Agent: iss-tracker


In [21]:
def ask(question):
    conv = openai_client.conversations.create()
    resp = openai_client.responses.create(
        conversation=conv.id, input=question,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}}
    )
    approvals = [McpApprovalResponse(type="mcp_approval_response", approval_request_id=i.id, approve=True) 
                 for i in resp.output if i.type == "mcp_approval_request" and i.id]
    if approvals:
        resp = openai_client.responses.create(
            input=approvals, previous_response_id=resp.id,
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}}
        )
    return resp.output_text

print(ask("Where is the ISS right now?"))

Right now, the ISS is located at approximately latitude -34.13 and longitude -150.44, at an altitude of about 408 kilometers above the Earth.


In [22]:
print(ask("Can I see the ISS from Seattle (47.6, -122.3)?"))

The ISS is currently not visible from Seattle (latitude 47.6, longitude -122.3). It is about 9303 km away in the southwest direction. If you'd like, I can let you know when it will be visible from there.


In [23]:
print(ask("What are the ISS orbital parameters?"))

The orbital parameters of the International Space Station (ISS) are as follows:
- Altitude: 408 km
- Orbital period: 92.65 minutes
- Inclination: 51.6 degrees
- Velocity: 7.66 km/s
- Orbits per day: 15.54


## Cleanup

In [24]:
# Uncomment to delete resources
# !az functionapp delete -g {RESOURCE_GROUP} -n {FUNC_APP_NAME} -y
# !az storage account delete -g {RESOURCE_GROUP} -n {STORAGE_NAME} -y